In [0]:
# ═══════════════════════════════════════════════
# 04_DATA_QUALITY — Validate the Silver table
# ═══════════════════════════════════════════════
from pyspark.sql.functions import col, count, when

silver = spark.table("silver_card_spending")
print("DATA QUALITY CHECKS")

# Check 1: Row count
row_count = silver.count()
print(f"1. Row count: {row_count}", "PASS" if row_count > 0 else "FAIL")

# Check 2: Nulls in key columns
nulls = silver.select(
    count(when(col("date").isNull(), 1)).alias("d"),
    count(when(col("spend_index").isNull(), 1)).alias("s"),
    count(when(col("category").isNull(), 1)).alias("c")
).collect()[0]
total_nulls = nulls["d"] + nulls["s"] + nulls["c"]
print(f"2. Null count: {total_nulls}", "PASS" if total_nulls == 0 else "FAIL")

# Check 3: Duplicates
dupes = silver.count() - silver.select("date", "category").distinct().count()
print(f"3. Duplicates: {dupes}", "PASS" if dupes == 0 else "WARNING")

# Check 4: Category integrity
cats = {r["category"] for r in silver.select("category").distinct().collect()}
expected = {"Social", "Staple", "Delayable", "Work Related"}
print(f"4. Categories: {cats}", "PASS" if cats == expected else "CHECK")

# Fail the task loudly if critical checks fail (so the Workflow catches it)
assert row_count > 0, "No data in Silver!"
assert total_nulls == 0, "Nulls found in key columns!"
print("\nAll critical DQ checks passed.")

DATA QUALITY CHECKS
1. Row count: 4184 PASS
2. Null count: 0 PASS
3. Duplicates: 0 PASS
4. Categories: {'Delayable', 'Work Related', 'Staple', 'Social'} PASS

All critical DQ checks passed.
